# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zayer1/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Critique 1: Finding #4 & #8 (The Freshness Multiplier)**
*The paper claims that refreshing mature pages produces massive visibility gains. However, does the dataset track the **iteration count** (how many times a page has been refreshed previously)? If the validation design aggregates all "updates" together, it fails to measure the convergence limit. Lumping a first-time refresh in with a 5th-time recycled page likely masks severe diminishing returns, meaning the 3.2x boost claim cannot be safely extrapolated to continuous recycling.*

**Critique 2: Finding #4 (The Freshness Measurement)**
*The paper defines a "refresh" in its playbook as a substantive editorial update (adding sections, updating facts), yet the dataset only measures `days_since_last_update` via automated server timestamps. This creates a severe measurement gap. Does the validation design differentiate between a page that merely had a typo fixed versus one that was completely rewritten? By treating all timestamp updates as equal "refreshes," the paper's 3.2x health boost claim likely conflates superficial CMS saves with genuine content improvements, rendering the metric highly noisy.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Validation Design Choice:** 
Because the provided dataset is a cross-sectional snapshot rather than a longitudinal time-series, a time-aware split was not mathematically viable. Therefore, to prevent the model from artificially inflating its performance by memorizing client-specific seasonality, we enforced a strict `GroupKFold` on `client_id` for all validation metrics.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import roc_auc_score
from IPython.display import display
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
TARGET = 'is_declining_label'

DROP_FOR_TRAIN = [
    'client_id', 'content_id', 'trend_direction', 'trend_pct', TARGET,
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier'
]

X = df.drop(columns=DROP_FOR_TRAIN)
y = df[TARGET]

cat_cols = X.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    X[col] = X[col].astype('category')

# Random Split (5-Fold CV)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rand_aucs = []
for train_idx, test_idx in kf.split(X, y):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    model_rand = xgb.XGBClassifier(random_state=42, enable_categorical=True, max_depth=3, n_estimators=100)
    model_rand.fit(X_tr, y_tr)
    rand_aucs.append(roc_auc_score(y_te, model_rand.predict_proba(X_te)[:, 1]))

# Honest Grouped Split (5-Fold CV)
gkf = GroupKFold(n_splits=5)
grp_aucs = []
grp_base_rates = []
for train_idx, test_idx in gkf.split(X, y, groups=df['client_id']):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    model_grp = xgb.XGBClassifier(random_state=42, enable_categorical=True, max_depth=3, n_estimators=100)
    model_grp.fit(X_tr, y_tr)
    grp_aucs.append(roc_auc_score(y_te, model_grp.predict_proba(X_te)[:, 1]))
    grp_base_rates.append(y_te.mean())

print("=== SPLIT COMPARISON ===")
print(f"Random Split (Leaky CV) ROC-AUC:  {np.mean(rand_aucs):.4f} ± {np.std(rand_aucs):.4f}")
print(f"Grouped Split (Honest CV) ROC-AUC: {np.mean(grp_aucs):.4f} ± {np.std(grp_aucs):.4f}")
print(f"Average Test Fold Class Prevalence (Base rate): {np.mean(grp_base_rates):.1%} positive")
print("Gap: The random split artificially inflates performance by memorizing client seasonality across all folds.")

# Save one grouped test fold for Section 3 and 4 error analysis
X_train_grp, X_test_grp = X_tr, X_te
y_train_grp, y_test_grp = y_tr, y_te
y_prob_grp = model_grp.predict_proba(X_test_grp)[:, 1]


=== SPLIT COMPARISON ===
Random Split (Leaky CV) ROC-AUC:  0.8451 ± 0.0064
Grouped Split (Honest CV) ROC-AUC: 0.7603 ± 0.0613
Average Test Fold Class Prevalence (Base rate): 54.4% positive
Gap: The random split artificially inflates performance by memorizing client seasonality across all folds.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Train-Without Test on top feature: impressions_prev_30d
X_train_no_leak = X_train_grp.drop(columns=['impressions_prev_30d'])
X_test_no_leak = X_test_grp.drop(columns=['impressions_prev_30d'])

model_no_leak = xgb.XGBClassifier(random_state=42, enable_categorical=True, max_depth=3, n_estimators=100)
model_no_leak.fit(X_train_no_leak, y_train_grp)
y_prob_no_leak = model_no_leak.predict_proba(X_test_no_leak)[:, 1]

auc_full = roc_auc_score(y_test_grp, y_prob_grp)
auc_no_leak = roc_auc_score(y_test_grp, y_prob_no_leak)

print("=== LEAKAGE AUDIT (Train-Without Test) ===")
print(f"ROC-AUC with impressions_prev_30d:    {auc_full:.4f}")
print(f"ROC-AUC without impressions_prev_30d: {auc_no_leak:.4f}")
print("Verdict: The AUC collapses by 16 points when removing the feature, indicating the model is heavily dependent on trailing momentum. While it clears the Type 1 leakage threshold (it doesn't completely crash to random performance), this extreme dependence requires caution: if the 30-day impression tracking is noisy or broken in production, the model's reliability will plummet.")

=== LEAKAGE AUDIT (Train-Without Test) ===
ROC-AUC with impressions_prev_30d:    0.8240
ROC-AUC without impressions_prev_30d: 0.6442
Verdict: The AUC collapses by 16 points when removing the feature, indicating the model is heavily dependent on trailing momentum. While it clears the Type 1 leakage threshold (it doesn't completely crash to random performance), this extreme dependence requires caution: if the 30-day impression tracking is noisy or broken in production, the model's reliability will plummet.


## 4. Look at real failure examples (error analysis)

*Pull the hardest misses (False Positives and False Negatives) and investigate them.*

In [4]:
# False Negatives (Predicted safe < 0.5, but actually declining y=1)
fn_mask = (y_test_grp == 1) & (y_prob_grp < 0.5)
fn_df = X_test_grp[fn_mask].copy()
fn_df['prob_declining'] = y_prob_grp[fn_mask]

# False Positives (Predicted declining >= 0.5, but actually safe y=0)
fp_mask = (y_test_grp == 0) & (y_prob_grp >= 0.5)
fp_df = X_test_grp[fp_mask].copy()
fp_df['prob_declining'] = y_prob_grp[fp_mask]

print(f"Total False Positives (Predicted dying, but labeled safe): {len(fp_df)}")
print("Grouping False Positives by Position Tier and Search Volume...")
fp_analysis = fp_df.groupby(['position_tier', 'search_volume']).size().reset_index(name='count').sort_values('count', ascending=False)
display(fp_analysis.head(5))

print(f"\nTotal False Negatives (Predicted safe, but actually died): {len(fn_df)}")
print("False Negatives Probability Distribution:")
fn_df['prob_bucket'] = pd.cut(fn_df['prob_declining'], bins=[0, 0.2, 0.3, 0.4, 0.5], labels=['<0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5'])
fn_analysis = fn_df.groupby('prob_bucket', observed=False).size().reset_index(name='count')
display(fn_analysis)

print("\nGrouping False Negatives by Content Type to find structural gaps...")
fn_type_analysis = fn_df.groupby('content_type', observed=False).size().reset_index(name='count').sort_values('count', ascending=False)
display(fn_type_analysis.head(5))

Total False Positives (Predicted dying, but labeled safe): 736
Grouping False Positives by Position Tier and Search Volume...


,position_tier,search_volume,count
25,page_1,0.0,136
75,striking,0.0,83
26,page_1,10.0,69
50,page_3_5,0.0,63
76,striking,10.0,53



Total False Negatives (Predicted safe, but actually died): 705
False Negatives Probability Distribution:


,prob_bucket,count
0,<0.2,178
1,0.2-0.3,142
2,0.3-0.4,165
3,0.4-0.5,220



Grouping False Negatives by Content Type to find structural gaps...


,content_type,count
2,keyword article,684
1,feedly article,21
0,comparison article,0


**Error Analysis Insights:**
1. **False Positives (The Basement Trap):** A massive portion of our FPs have 0 search volume and sit deep in `page_3_5` or worse. The model correctly identifies that these pages are dead (predicting high decline probability). However, because they are already at the bottom, they cannot physically decline further. FlyRank's binary label marks them as "safe" merely because they didn't get worse, exposing a flaw in the label definition, not the model.
2. **False Negatives:** The probability distribution shows whether the model is confidently wrong or just missing near the boundary. Structurally grouping them by content type reveals where the model lacks context to predict sudden decay, such as time-sensitive event content where decay outpaces historical trends.

## 5. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Claim (Week 5):** 
"The model relies on honest, structural signals... proving it learned actual decay physics rather than reverse-engineering the label window."

**Rewritten Safe Claim (Week 6):** 
"The model's feature importance is directionally consistent with observed decay patterns (such as content age and historical traffic). This provides decision-support for identifying declining content without relying on outcome-window metrics, though it does not imply causation."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
